# Task 4: Video-Aware Data Splitting & Optimization

**Project:** Cognitive Fire Defense Pipeline — AIN7601  
**Goal:** Mathematically optimal 70/15/15 clip-level split to prevent temporal leakage, while balancing data distributions (blur, brightness, box area) across splits.


In [4]:
%pip install tqdm


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Configuration

import os
import glob
import random
import shutil
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

# Dataset B — CLEANED DATA
DATA_B = "../../working-Boreal-Forest-Fire-Subset-A"
# Output directory
YOLO_DIR = "../../yolo_format"

# Maximum frames retained from one clip
MAX_FRAMES_PER_CLIP = 300

# Optimization
OPTIMIZATION_ITERATIONS = 10000

# Train / Validation only
TRAIN_RATIO = 0.80
VAL_RATIO = 0.20

locations = [
    "Evo",
    "Heinola",
    "Karkkila",
    "Ruokolahti"
]

# Create output directories
for split in ["train", "val"]:
    os.makedirs(
        os.path.join(YOLO_DIR, "images", split),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(YOLO_DIR, "labels", split),
        exist_ok=True
    )

print("Configuration loaded.")
print(f"Train ratio: {TRAIN_RATIO}")
print(f"Validation ratio: {VAL_RATIO}")
print(f"Max frames per clip: {MAX_FRAMES_PER_CLIP}")

Configuration loaded.
Train ratio: 0.8
Validation ratio: 0.2
Max frames per clip: 300


## 1. Discover Clips & Empty Images
We will group images by their video prefix.

In [ ]:
# Load Images and Group into Clips

all_images = []

for loc in locations:

    img_dir = os.path.join(
        DATA_B,
        f"{loc}-Images"
    )

    if os.path.exists(img_dir):

        images = glob.glob(
            os.path.join(img_dir, "*.jpg")
        )

        all_images.extend(images)

        print(
            f"{loc}: {len(images)} images"
        )

    else:
        print(
            f"WARNING: Missing directory: {img_dir}"
        )


print("\nTotal images:", len(all_images))

# Group frames by clip ID

clips = defaultdict(list)

for img_path in all_images:

    filename = os.path.basename(img_path)

    # Example:
    # evoDJI_0001_frame198.jpg
    #
    # -> evoDJI_0001

    if "_frame" not in filename:
        print(
            f"WARNING: Cannot extract clip ID: {filename}"
        )
        continue

    seq_id = filename.split("_frame")[0]

    clips[seq_id].append(img_path)


# Sort frames chronologically
for seq_id in clips:

    clips[seq_id].sort(
        key=lambda x: int(
            os.path.basename(x)
            .split("_frame")[1]
            .split(".")[0]
        )
    )


print("\nTotal clips:", len(clips))

for seq_id, paths in sorted(clips.items()):

    print(
        f"{seq_id}: {len(paths)} frames"
    )

Evo: 931 images
Heinola: 906 images
Karkkila: 1096 images
Ruokolahti: 1765 images

Total images: 4698

Total clips: 30
evoDJI_0001: 314 frames
evoDJI_0007: 328 frames
evoDJI_0009: 289 frames
heinola_DJI_0025: 201 frames
heinola_DJI_0028: 156 frames
heinola_DJI_0029: 106 frames
heinola_DJI_0030: 143 frames
heinola_DJI_0033: 230 frames
heinola_DJI_0036: 45 frames
heinola_DJI_0038: 10 frames
heinola_DJI_0048: 15 frames
karkkila_DJI_0001: 69 frames
karkkila_DJI_0002: 125 frames
karkkila_DJI_0003: 26 frames
karkkila_DJI_0004: 71 frames
karkkila_DJI_0005: 171 frames
karkkila_DJI_0006: 100 frames
karkkila_DJI_0007: 171 frames
karkkila_DJI_0008: 171 frames
karkkila_DJI_0010: 58 frames
karkkila_DJI_0020: 134 frames
ruokolahti_DJI_0080: 341 frames
ruokolahti_DJI_0081: 128 frames
ruokolahti_DJI_0082: 70 frames
ruokolahti_DJI_0084: 23 frames
ruokolahti_DJI_0085: 176 frames
ruokolahti_DJI_0086: 328 frames
ruokolahti_DJI_0087: 282 frames
ruokolahti_DJI_0088: 328 frames
ruokolahti_DJI_0089: 89 frames

In [ ]:
# CELL 5 — Extract Clip-Level Features

def extract_clip_features(img_paths):

    # --------------------------------------------------------
    # Pixel-based features
    # Sample up to 10 images only for expensive processing
    # --------------------------------------------------------

    sample_paths = random.sample(
        img_paths,
        min(10, len(img_paths))
    )

    brightness = []
    blur = []

    for img_p in sample_paths:

        img = cv2.imread(
            img_p,
            cv2.IMREAD_GRAYSCALE
        )

        if img is not None:

            small_img = cv2.resize(
                img,
                (512, 512)
            )

            brightness.append(
                np.mean(small_img)
            )

            blur.append(
                cv2.Laplacian(
                    small_img,
                    cv2.CV_64F
                ).var()
            )


    # --------------------------------------------------------
    # Label-based features
    # IMPORTANT:
    # Parse ALL frames, not only the 10 sampled images.
    # --------------------------------------------------------

    box_areas = []

    small_plumes = 0
    total_boxes = 0

    for img_p in img_paths:

        lbl_p = (
            img_p
            .replace("-Images", "-Labels")
            .replace(".jpg", ".txt")
        )

        if not os.path.exists(lbl_p):
            continue

        with open(lbl_p, "r") as f:

            for line in f:

                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                _, _, _, w, h = map(
                    float,
                    parts
                )

                area = w * h

                box_areas.append(area)

                if area < 0.01:
                    small_plumes += 1

                total_boxes += 1


    # Location
    filename = os.path.basename(
        img_paths[0]
    ).lower()

    if filename.startswith("evo"):
        location = "Evo"

    elif filename.startswith("heinola"):
        location = "Heinola"

    elif filename.startswith("karkkila"):
        location = "Karkkila"

    elif filename.startswith("ruokolahti"):
        location = "Ruokolahti"

    else:
        location = "Unknown"

    # Return

    return {

        # Actual number of frames available,
        # capped only for optimization/splitting.
        "count": min(
            len(img_paths),
            MAX_FRAMES_PER_CLIP
        ),

        "brightness": (
            np.mean(brightness)
            if brightness
            else 0
        ),

        "blur": (
            np.mean(blur)
            if blur
            else 0
        ),

        "box_area": (
            np.mean(box_areas)
            if box_areas
            else 0
        ),

        "small_plume_ratio": (
            small_plumes / total_boxes
            if total_boxes > 0
            else 0
        ),

        "location": location
    }

# Extract features

print(
    f"Extracting features for {len(clips)} clips..."
)

clip_data = {}

for seq_id, paths in tqdm(
    clips.items()
):

    clip_data[seq_id] = extract_clip_features(
        paths
    )


print(
    f"\nFeature extraction completed: "
    f"{len(clip_data)} clips"
)

Extracting features for 30 clips...


100%|██████████| 30/30 [00:10<00:00,  3.00it/s]


Feature extraction completed: 30 clips


In [8]:
# Distribution by location
from collections import Counter

locations_count = Counter()

for seq_id in clips.keys():
    seq_lower = seq_id.lower()

    if seq_lower.startswith("evo"):
        locations_count["Evo"] += 1
    elif seq_lower.startswith("heinola"):
        locations_count["Heinola"] += 1
    elif seq_lower.startswith("karkkila"):
        locations_count["Karkkila"] += 1
    elif seq_lower.startswith("ruokolahti"):
        locations_count["Ruokolahti"] += 1
    else:
        locations_count["Unknown"] += 1

print(locations_count)

Counter({'Karkkila': 10, 'Ruokolahti': 9, 'Heinola': 8, 'Evo': 3})


In [9]:
# Total frames

total_frames = sum(len(paths) for paths in clips.values())

print(f"Total frames across clips: {total_frames}")
print(f"Original images: {len(all_images)}")

Total frames across clips: 4698
Original images: 4698


In [10]:
clip_ids = list(clips.keys())

print(f"Number of clip IDs: {len(clip_ids)}")
print(f"Number of unique clip IDs: {len(set(clip_ids))}")

if len(clip_ids) == len(set(clip_ids)):
    print("✓ No duplicate clip IDs")
else:
    print("⚠ Duplicate clip IDs found")

Number of clip IDs: 30
Number of unique clip IDs: 30
✓ No duplicate clip IDs


## 2. Fast Clip Feature Extraction
To balance the splits, we need to know the representative properties of each clip. To avoid 4K processing bottlenecks, we randomly sample up to 10 frames per clip to estimate its Mean Brightness, Mean Blur, Mean Box Area, and Small Plume %.


In [14]:
# Task 4 Configuration

MAX_FRAMES_PER_CLIP = 300
OPTIMIZATION_ITERATIONS = 10000

print("MAX_FRAMES_PER_CLIP =", MAX_FRAMES_PER_CLIP)
print("OPTIMIZATION_ITERATIONS =", OPTIMIZATION_ITERATIONS)

MAX_FRAMES_PER_CLIP = 300
OPTIMIZATION_ITERATIONS = 10000


In [15]:
def extract_clip_features(img_paths):

    # Sample up to 10 images for expensive pixel calculations
    sample_paths = random.sample(
        img_paths,
        min(10, len(img_paths))
    )

    brightness = []
    blur = []

    # 1. Brightness and Blur
    for img_p in sample_paths:

        img = cv2.imread(
            img_p,
            cv2.IMREAD_GRAYSCALE
        )

        if img is not None:

            small_img = cv2.resize(
                img,
                (512, 512)
            )

            brightness.append(
                np.mean(small_img)
            )

            blur.append(
                cv2.Laplacian(
                    small_img,
                    cv2.CV_64F
                ).var()
            )


    # 2. Bounding Box Statistics - Process ALL images in the clip

    box_areas = []
    small_plumes = 0
    total_boxes = 0

    for img_p in img_paths:

        # Build corresponding label path
        lbl_p = (
            img_p
            .replace("Images", "Labels")
            .replace(".jpg", ".txt")
        )

        if os.path.exists(lbl_p):

            with open(lbl_p, "r") as f:

                for line in f.readlines():

                    parts = line.strip().split()

                    if len(parts) == 5:

                        _, _, _, w, h = map(
                            float,
                            parts
                        )

                        area = w * h

                        box_areas.append(area)

                        if area < 0.01:
                            small_plumes += 1

                        total_boxes += 1

    # 3. Determine Location

    first_filename = os.path.basename(
        img_paths[0]
    )

    first_part = first_filename.split("_")[0]

    if first_part.lower().startswith("evo"):
        location = "Evo"

    elif first_part.lower().startswith("heinola"):
        location = "Heinola"

    elif first_part.lower().startswith("karkkila"):
        location = "Karkkila"

    elif first_part.lower().startswith("ruokolahti"):
        location = "Ruokolahti"

    else:
        location = "Unknown"

    # 4. Return Features

    return {
        "count": min(
            len(img_paths),
            MAX_FRAMES_PER_CLIP
        ),

        "brightness": (
            np.mean(brightness)
            if brightness
            else 0
        ),

        "blur": (
            np.mean(blur)
            if blur
            else 0
        ),

        "box_area": (
            np.mean(box_areas)
            if box_areas
            else 0
        ),

        "small_plume_ratio": (
            small_plumes / total_boxes
            if total_boxes > 0
            else 0
        ),

        "location": location
    }


# Extract features for all clips

print(
    f"Extracting features for {len(clips)} clips..."
)

clip_data = {}

for seq_id, paths in tqdm(
    clips.items()
):

    clip_data[seq_id] = extract_clip_features(
        paths
    )

Extracting features for 30 clips...


100%|██████████| 30/30 [00:09<00:00,  3.01it/s]


In [26]:
# check function
print(f"Number of clips: {len(clip_data)}")

for seq_id, features in clip_data.items():
    print(
        seq_id,
        "->",
        features
    )

Number of clips: 30
evoDJI_0009 -> {'count': 289, 'brightness': np.float64(96.18756065368652), 'blur': np.float64(252.7155222699992), 'box_area': np.float64(0.36587216900643943), 'small_plume_ratio': 0.0, 'location': 'Evo'}
evoDJI_0001 -> {'count': 300, 'brightness': np.float64(100.60015716552735), 'blur': np.float64(2339.8195357972754), 'box_area': np.float64(0.3345552764627186), 'small_plume_ratio': 0.027522935779816515, 'location': 'Evo'}
evoDJI_0007 -> {'count': 300, 'brightness': np.float64(98.7634334564209), 'blur': np.float64(2478.8708551147124), 'box_area': np.float64(0.34661286996944873), 'small_plume_ratio': 0.02346041055718475, 'location': 'Evo'}
heinola_DJI_0033 -> {'count': 230, 'brightness': np.float64(109.91879310607911), 'blur': np.float64(2211.4307896239593), 'box_area': np.float64(0.2844544245992681), 'small_plume_ratio': 0.01276595744680851, 'location': 'Heinola'}
heinola_DJI_0030 -> {'count': 143, 'brightness': np.float64(93.45473937988281), 'blur': np.float64(1223.

## 3. Stochastic Optimization Packing
We run a randomized assignment 5,000 times to find the split that minimizes deviation from the 70/15/15 ratio AND minimizes the variance of Brightness, Blur, Box Area, and Small Plumes across Train/Val/Test.


In [ ]:

# Train / Validation Clip-Level Optimization

best_score = float("inf")
best_split = None

splits = ["train", "val"]


# Total frames used by optimizer


total_frames = sum(
    c["count"]
    for c in clip_data.values()
)


target_train = (
    total_frames * TRAIN_RATIO
)

target_val = (
    total_frames * VAL_RATIO
)


print("Total optimization frames:", total_frames)
print("Target Train:", target_train)
print("Target Val:", target_val)


# Optimization

for iteration in range(
    OPTIMIZATION_ITERATIONS
):

    assignment = {
        seq_id: random.choice(splits)
        for seq_id in clip_data
    }

    # Location constraint

    loc_splits = {
        "train": set(),
        "val": set()
    }

    for seq_id, split in assignment.items():

        loc_splits[split].add(
            clip_data[seq_id]["location"]
        )


    # Both splits must contain all locations
    if any(
        len(loc_splits[split]) < len(locations)
        for split in splits
    ):
        continue


    # Calculate frame counts

    counts = {
        "train": 0,
        "val": 0
    }

    metrics = {
        "train": [],
        "val": []
    }

    for seq_id, split in assignment.items():

        c = clip_data[seq_id]

        counts[split] += c["count"]

        metrics[split].append(c)


    if counts["train"] == 0:
        continue

    if counts["val"] == 0:
        continue

    # Ratio penalty

    ratio_penalty = (

        abs(
            counts["train"] /
            total_frames -
            TRAIN_RATIO
        )

        +

        abs(
            counts["val"] /
            total_frames -
            VAL_RATIO
        )
    )

    # Distribution penalty

    dist_penalty = 0

    for key in [
        "brightness",
        "blur",
        "box_area",
        "small_plume_ratio"
    ]:

        vals = []

        for split in splits:

            split_val = (
                sum(
                    c[key] * c["count"]
                    for c in metrics[split]
                )
                /
                counts[split]
            )

            vals.append(split_val)


        mean_val = np.mean(vals)

        if mean_val > 0:

            dist_penalty += (
                np.var(vals) /
                (mean_val ** 2)
            )

        else:

            dist_penalty += np.var(vals)

    # Small plume constraint

    for split in splits:

        small_plume_clips = sum(
            c["small_plume_ratio"] > 0
            for c in metrics[split]
        )

        if small_plume_clips == 0:

            dist_penalty += 100000.0


    # Blur balance

    blur_vals = []

    for split in splits:

        split_blur = (
            sum(
                c["blur"] * c["count"]
                for c in metrics[split]
            )
            /
            counts[split]
        )

        blur_vals.append(split_blur)


    dist_penalty += (
        np.std(blur_vals) * 10.0
    )

    # Final score

    score = (
        ratio_penalty * 100000
        +
        dist_penalty
    )


    if score < best_score:

        best_score = score

        best_split = assignment.copy()


print(
    f"\nOptimal Train/Val split found!"
)

print(
    f"Best score: {best_score:.4f}"
)

Total optimization frames: 4559
Target Train: 3647.2000000000003
Target Val: 911.8000000000001

Optimal Train/Val split found!
Best score: 2260.5239


In [ ]:
# Inspect Selected Split

for split in ["train", "val"]:

    selected_clips = [
        seq_id
        for seq_id, assigned_split
        in best_split.items()
        if assigned_split == split
    ]

    frame_count = sum(
        clip_data[seq_id]["count"]
        for seq_id in selected_clips
    )

    locations_present = sorted(
        set(
            clip_data[seq_id]["location"]
            for seq_id in selected_clips
        )
    )

    small_plume_clips = sum(
        clip_data[seq_id]["small_plume_ratio"] > 0
        for seq_id in selected_clips
    )

    print("\n" + "=" * 50)
    print(split.upper())
    print("=" * 50)

    print(
        "Clips:",
        len(selected_clips)
    )

    print(
        "Frames:",
        frame_count
    )

    print(
        "Ratio:",
        f"{frame_count / total_frames * 100:.2f}%"
    )

    print(
        "Locations:",
        locations_present
    )

    print(
        "Small-plume clips:",
        small_plume_clips
    )

    print(
        "\nClip IDs:"
    )

    for seq_id in sorted(selected_clips):
        print(
            f"  {seq_id}: "
            f"{len(clips[seq_id])} frames"
        )


TRAIN
Clips: 22
Frames: 3613
Ratio: 79.25%
Locations: ['Evo', 'Heinola', 'Karkkila', 'Ruokolahti']
Small-plume clips: 9

Clip IDs:
  evoDJI_0001: 314 frames
  evoDJI_0009: 289 frames
  heinola_DJI_0028: 156 frames
  heinola_DJI_0029: 106 frames
  heinola_DJI_0033: 230 frames
  heinola_DJI_0036: 45 frames
  heinola_DJI_0038: 10 frames
  heinola_DJI_0048: 15 frames
  karkkila_DJI_0001: 69 frames
  karkkila_DJI_0004: 71 frames
  karkkila_DJI_0005: 171 frames
  karkkila_DJI_0006: 100 frames
  karkkila_DJI_0007: 171 frames
  karkkila_DJI_0008: 171 frames
  karkkila_DJI_0020: 134 frames
  ruokolahti_DJI_0080: 341 frames
  ruokolahti_DJI_0081: 128 frames
  ruokolahti_DJI_0085: 176 frames
  ruokolahti_DJI_0086: 328 frames
  ruokolahti_DJI_0087: 282 frames
  ruokolahti_DJI_0088: 328 frames
  ruokolahti_DJI_0089: 89 frames

VAL
Clips: 8
Frames: 946
Ratio: 20.75%
Locations: ['Evo', 'Heinola', 'Karkkila', 'Ruokolahti']
Small-plume clips: 1

Clip IDs:
  evoDJI_0007: 328 frames
  heinola_DJI_0025: 

In [34]:
print("\n" + "=" * 60)
print("SPLIT VERIFICATION")
print("=" * 60)

for split in ["train", "val"]:

    clips_in_split = [
        seq
        for seq, s in best_split.items()
        if s == split
    ]

    frame_count = sum(
        clip_data[seq]["count"]
        for seq in clips_in_split
    )

    locations_in_split = sorted(
        set(
            clip_data[seq]["location"]
            for seq in clips_in_split
        )
    )

    small_plume_clips = [
        seq
        for seq in clips_in_split
        if clip_data[seq]["small_plume_ratio"] > 0
    ]

    print(f"\n{split.upper()}")
    print("-" * 40)

    print(f"Clips: {len(clips_in_split)}")
    print(f"Frames: {frame_count}")

    print(
        f"Ratio: "
        f"{frame_count / total_frames * 100:.2f}%"
    )

    print(f"Locations: {locations_in_split}")

    print(
        f"Small-plume clips: "
        f"{len(small_plume_clips)}"
    )


SPLIT VERIFICATION

TRAIN
----------------------------------------
Clips: 22
Frames: 3613
Ratio: 79.25%
Locations: ['Evo', 'Heinola', 'Karkkila', 'Ruokolahti']
Small-plume clips: 9

VAL
----------------------------------------
Clips: 8
Frames: 946
Ratio: 20.75%
Locations: ['Evo', 'Heinola', 'Karkkila', 'Ruokolahti']
Small-plume clips: 1


In [ ]:
# CHECK 1 — No Clip Overlap
train_clips = {
    seq for seq, split in best_split.items()
    if split == "train"
}

val_clips = {
    seq for seq, split in best_split.items()
    if split == "val"
}

overlap = train_clips.intersection(val_clips)

print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

print(f"Train clips: {len(train_clips)}")
print(f"Val clips: {len(val_clips)}")
print(f"Overlap: {len(overlap)}")

if len(overlap) == 0:
    print("✓ No clip overlap between Train and Validation")
else:
    print("✗ WARNING: Clip overlap detected!")
    print(overlap)


LEAKAGE CHECK
Train clips: 22
Val clips: 8
Overlap: 0
✓ No clip overlap between Train and Validation


In [ ]:
# CHECK 2 — Location Stratification

required_locations = set(locations)

train_locations = {
    clip_data[seq]["location"]
    for seq in train_clips
}

val_locations = {
    clip_data[seq]["location"]
    for seq in val_clips
}

print("\n" + "=" * 60)
print("LOCATION STRATIFICATION")
print("=" * 60)

print("Train:", sorted(train_locations))
print("Val:  ", sorted(val_locations))

if required_locations.issubset(train_locations):
    print("✓ All locations present in Train")
else:
    print("✗ Missing location in Train")

if required_locations.issubset(val_locations):
    print("✓ All locations present in Validation")
else:
    print("✗ Missing location in Validation")


LOCATION STRATIFICATION
Train: ['Evo', 'Heinola', 'Karkkila', 'Ruokolahti']
Val:   ['Evo', 'Heinola', 'Karkkila', 'Ruokolahti']
✓ All locations present in Train
✓ All locations present in Validation


In [ ]:
# CHECK 3 — Frame Retention

original_frames = sum(
    len(clips[seq])
    for seq in clip_data
)

retained_frames = sum(
    clip_data[seq]["count"]
    for seq in best_split
)

removed_frames = original_frames - retained_frames

print("\n" + "=" * 60)
print("FRAME RETENTION")
print("=" * 60)

print(f"Original frames:  {original_frames}")
print(f"Retained frames:  {retained_frames}")
print(f"Removed by cap:   {removed_frames}")

print(
    f"Retention: "
    f"{retained_frames / original_frames * 100:.2f}%"
)


FRAME RETENTION
Original frames:  4698
Retained frames:  4559
Removed by cap:   139
Retention: 97.04%


## 4. Execute Split & Apply Sampling
We will now copy the files, uniformly sampling frames for clips that exceed `MAX_FRAMES_PER_CLIP`.

In [35]:
final_metrics = {"train": defaultdict(list), "val": defaultdict(list), "test": defaultdict(list)}
total_copied = {"train": 0, "val": 0, "test": 0}

def copy_pair(img_src, split):
    lbl_src = img_src.replace("Images", "Labels").replace(".jpg", ".txt")
    
    # Standardize names safely
    base = os.path.basename(img_src)
    img_dst = os.path.join(YOLO_DIR, "images", split, base)
    lbl_dst = os.path.join(YOLO_DIR, "labels", split, base.replace(".jpg", ".txt"))
    
    shutil.copy2(img_src, img_dst)
    if os.path.exists(lbl_src):
        shutil.copy2(lbl_src, lbl_dst)
        
    total_copied[split] += 1

# Process Clips
for seq_id, split in best_split.items():
    paths = sorted(clips[seq_id])
    
    # Uniform linear sampling if exceeding max frames
    if len(paths) > MAX_FRAMES_PER_CLIP:
        indices = np.linspace(0, len(paths)-1, MAX_FRAMES_PER_CLIP, dtype=int)
        sampled_paths = [paths[i] for i in indices]
    else:
        sampled_paths = paths
        
    # Append to final metric calculations
    cd = clip_data[seq_id]
    for _ in range(len(sampled_paths)):
        final_metrics[split]["brightness"].append(cd["brightness"])
        final_metrics[split]["blur"].append(cd["blur"])
        final_metrics[split]["box_area"].append(cd["box_area"])
        final_metrics[split]["small_plume"].append(cd["small_plume_ratio"])
        
    for p in sampled_paths:
        copy_pair(p, split)

# Process Empty Images (Random Assignment at 70/15/15)
random.shuffle(empty_images)
num_empty = len(empty_images)
train_end = int(num_empty * 0.70)
val_end = train_end + int(num_empty * 0.15)

for i, p in enumerate(empty_images):
    if i < train_end:
        copy_pair(p, "train")
    elif i < val_end:
        copy_pair(p, "val")
    else:
        copy_pair(p, "test")

print("Files successfully copied to YOLO format!")


Files successfully copied to YOLO format!


## 5. Academically Defensible Distribution Report

In [6]:
from IPython.display import display, Markdown

# Calculate aggregates
data = []
for split in ["train", "val", "test"]:
    count = total_copied[split]
    ratio = count / sum(total_copied.values()) * 100
    
    b_mean = np.mean(final_metrics[split]["brightness"]) if final_metrics[split]["brightness"] else 0
    bl_mean = np.mean(final_metrics[split]["blur"]) if final_metrics[split]["blur"] else 0
    a_mean = np.mean(final_metrics[split]["box_area"]) if final_metrics[split]["box_area"] else 0
    sp_mean = np.mean(final_metrics[split]["small_plume"]) * 100 if final_metrics[split]["small_plume"] else 0
    
    data.append({
        "Split": split.capitalize(),
        "Images": count,
        "Ratio %": f"{ratio:.1f}%",
        "Mean Brightness": f"{b_mean:.1f}",
        "Mean Blur": f"{bl_mean:.1f}",
        "Mean Box Area": f"{a_mean:.3f}",
        "Small Plume %": f"{sp_mean:.1f}%"
    })

df_report = pd.DataFrame(data)

# Print as Markdown Table for the paper
md_table = df_report.to_markdown(index=False)
display(Markdown("### Distribution Metrics Across Splits"))
display(Markdown(md_table))

print("\n" + md_table)


### Distribution Metrics Across Splits

| Split   |   Images | Ratio %   |   Mean Brightness |   Mean Blur |   Mean Box Area | Small Plume %   |
|:--------|---------:|:----------|------------------:|------------:|----------------:|:----------------|
| Train   |     3066 | 63.7%     |             113.9 |      2710.1 |           0.454 | 1.5%            |
| Val     |      926 | 19.2%     |             109.6 |      1852   |           0.419 | 0.4%            |
| Test    |      823 | 17.1%     |             109.2 |      2749.6 |           0.339 | 1.0%            |


| Split   |   Images | Ratio %   |   Mean Brightness |   Mean Blur |   Mean Box Area | Small Plume %   |
|:--------|---------:|:----------|------------------:|------------:|----------------:|:----------------|
| Train   |     3066 | 63.7%     |             113.9 |      2710.1 |           0.454 | 1.5%            |
| Val     |      926 | 19.2%     |             109.6 |      1852   |           0.419 | 0.4%            |
| Test    |      823 | 17.1%     |             109.2 |      2749.6 |           0.339 | 1.0%            |
